In [ ]:
# ================================
# DATA AUGMENTATION SCRIPT - 6 VARIANTS
# ================================

import os
import cv2
import numpy as np
from PIL import Image, ImageEnhance
import random
import shutil
from pathlib import Path

class DataAugmentator:
    def __init__(self, output_base="data_augmentation_malo2"):
        self.output_base = output_base
        self.supported_formats = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    
    def create_output_structure(self, input_folders):
        for folder_path in input_folders:
            folder_name = os.path.basename(folder_path)
            output_folder = os.path.join(self.output_base, folder_name)
            os.makedirs(output_folder, exist_ok=True)
            print(f"Created output folder: {output_folder}")
    
    def flip_rotate(self, image):
        if random.random() > 0.5:
            image = cv2.flip(image, 1)
        if random.random() > 0.5:
            image = cv2.flip(image, 0)
        
        rotation_angle = random.choice([0, 90, 180, 270])
        if rotation_angle != 0:
            h, w = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)
            image = cv2.warpAffine(image, M, (w, h))
        return image
    
    def color_jittering(self, image):
        pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        brightness_factor = random.uniform(0.7, 1.3)
        pil_image = ImageEnhance.Brightness(pil_image).enhance(brightness_factor)
        
        contrast_factor = random.uniform(0.8, 1.2)
        pil_image = ImageEnhance.Contrast(pil_image).enhance(contrast_factor)
        
        saturation_factor = random.uniform(0.8, 1.2)
        pil_image = ImageEnhance.Color(pil_image).enhance(saturation_factor)
        
        return cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    
    def pixelation(self, image):
        h, w = image.shape[:2]
        pixel_factor = random.randint(2, 8)
        
        small_image = cv2.resize(image, (w // pixel_factor, h // pixel_factor), 
                               interpolation=cv2.INTER_LINEAR)
        pixelated_image = cv2.resize(small_image, (w, h), 
                                   interpolation=cv2.INTER_NEAREST)
        return pixelated_image
    
    def random_cropping(self, image):
        h, w = image.shape[:2]
        crop_factor = random.uniform(0.7, 0.95)
        crop_h = int(h * crop_factor)
        crop_w = int(w * crop_factor)
        
        start_h = random.randint(0, h - crop_h)
        start_w = random.randint(0, w - crop_w)
        
        cropped = image[start_h:start_h + crop_h, start_w:start_w + crop_w]
        resized = cv2.resize(cropped, (w, h))
        return resized
    
    #deshabilitado el cambio de color para reconocimeinto de buena calidad
    def color_inversion(self, image):
        """
        FIXED VERSION - No more overflow errors
        """
        variant = random.choice(['invert', 'hue_shift', 'channel_shift'])
        
        try:
            if variant == 'invert':
                return 255 - image
            
            elif variant == 'hue_shift':
                # SAFE HSV conversion
                hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
                hsv_float = hsv.astype(np.float32)
                hue_shift = random.randint(-30, 30)
                hsv_float[:, :, 0] = (hsv_float[:, :, 0] + hue_shift) % 180
                hsv_safe = np.clip(hsv_float, 0, 255).astype(np.uint8)
                return cv2.cvtColor(hsv_safe, cv2.COLOR_HSV2BGR)
            
            else:  # channel_shift
                b, g, r = cv2.split(image)
                channels = [b, g, r]
                random.shuffle(channels)
                return cv2.merge(channels)
                
        except Exception as e:
            print(f"Color inversion failed, using original image: {e}")
            return image
    
    def width_height_shift(self, image):
        h, w = image.shape[:2]
        width_factor = random.uniform(0.8, 1.2)
        height_factor = random.uniform(0.8, 1.2)
        
        new_w = int(w * width_factor)
        new_h = int(h * height_factor)
        
        stretched = cv2.resize(image, (new_w, new_h))
        final = cv2.resize(stretched, (w, h))
        return final
    
    def apply_augmentation(self, image, variant_name):
        if variant_name == "flip_rotate":
            return self.flip_rotate(image)
        elif variant_name == "color_jittering":
            return self.color_jittering(image)
        elif variant_name == "pixelation":
            return self.pixelation(image)
        elif variant_name == "random_cropping":
            return self.random_cropping(image)
        elif variant_name == "color_inversion":
            return self.color_inversion(image)
        elif variant_name == "width_height_shift":
            return self.width_height_shift(image)
        else:
            return image
    
    def process_folders(self, input_folders, copies_per_variant=3):
        variants = [
            "flip_rotate",
            "color_jittering", 
            "pixelation",
            "random_cropping",
            "color_inversion",
            "width_height_shift"
        ]
        
        if isinstance(copies_per_variant, int):
            variant_copies = {variant: copies_per_variant for variant in variants}
        else:
            variant_copies = copies_per_variant
        
        self.create_output_structure(input_folders)
        
        for folder_path in input_folders:
            print(f"\nProcessing folder: {folder_path}")
            folder_name = os.path.basename(folder_path)
            output_folder = os.path.join(self.output_base, folder_name)
            
            image_files = []
            for ext in self.supported_formats:
                image_files.extend(Path(folder_path).glob(f"*{ext}"))
                image_files.extend(Path(folder_path).glob(f"*{ext.upper()}"))
            
            print(f"Found {len(image_files)} images")
            
            for img_file in image_files:
                shutil.copy2(img_file, output_folder)
            
            for variant in variants:
                copies = variant_copies.get(variant, 0)
                if copies > 0:
                    print(f"  Applying {variant}: {copies} copies per image")
                    
                    for img_file in image_files:
                        image = cv2.imread(str(img_file))
                        if image is None:
                            continue
                        
                        for copy_num in range(copies):
                            try:
                                augmented = self.apply_augmentation(image, variant)
                                base_name = img_file.stem
                                extension = img_file.suffix
                                output_name = f"{base_name}_{variant}_{copy_num + 1}{extension}"
                                output_path = os.path.join(output_folder, output_name)
                                cv2.imwrite(output_path, augmented)
                            except Exception as e:
                                print(f"Error processing {img_file} with {variant}: {e}")
            
            final_count = len(list(Path(output_folder).glob("*")))
            print(f"  Output: {final_count} total images in {output_folder}")
    
    def get_statistics(self):
        if not os.path.exists(self.output_base):
            print("No augmented data found")
            return
        
        print(f"\nDATA AUGMENTATION STATISTICS:")
        print("=" * 50)
        
        total_images = 0
        for folder_name in os.listdir(self.output_base):
            folder_path = os.path.join(self.output_base, folder_name)
            if os.path.isdir(folder_path):
                image_count = len([f for f in os.listdir(folder_path) 
                                 if any(f.lower().endswith(ext) for ext in self.supported_formats)])
                print(f"{folder_name}: {image_count} images")
                total_images += image_count
        
        print(f"\nTotal augmented images: {total_images}")

# ================================
# ONLY THE QUICK FUNCTION
# ================================

def quick_augmentation(input_folders, copies=3):
    """
    Auto-generate output folder and process augmentation
    
    Example:
        quick_augmentation(['./dataset/xbox_joystick/'], copies=3)
        # Creates: './dataset/xbox_joystick/-data-augmentation'
    """
    if isinstance(input_folders, str):
        input_folders = [input_folders]
    
    first_folder = input_folders[0].rstrip('/')
    output_folder = f"{first_folder}-data-augmentation"
    
    print(f"Input: {input_folders}")
    print(f"Output: {output_folder}")
    
    augmentator = DataAugmentator(output_base=output_folder)
    augmentator.process_folders(input_folders, copies_per_variant=copies)
    augmentator.get_statistics()
    print(f"Augmentation completed! Check: {output_folder}/")

# Usage:
quick_augmentation(['./dataset/mis_imagenes_botella_buena/'], copies=3)

Input: ['./dataset/mis_imagenes_tapa_buena/']
Output: ./dataset/mis_imagenes_tapa_buena-data-augmentation
Created output folder: ./dataset/mis_imagenes_tapa_buena-data-augmentation\

Processing folder: ./dataset/mis_imagenes_tapa_buena/
Found 18 images
  Applying flip_rotate: 3 copies per image
  Applying color_jittering: 3 copies per image
  Applying pixelation: 3 copies per image
  Applying random_cropping: 3 copies per image
  Applying color_inversion: 3 copies per image
  Applying width_height_shift: 3 copies per image
  Output: 171 total images in ./dataset/mis_imagenes_tapa_buena-data-augmentation\

DATA AUGMENTATION STATISTICS:

Total augmented images: 0
Augmentation completed! Check: ./dataset/mis_imagenes_tapa_buena-data-augmentation/
